# POI Ingestion - Bronze Layer

Downloads OpenStreetMap data from Geofabrik and extracts POI nodes into Unity Catalog.

**Data Source:** Geofabrik OSM Extracts (PBF format)

**Output Table:**
- `{catalog}.{bronze_schema}.raw_pois` - Raw POI data with tags

In [ ]:
# MAGIC %md
# MAGIC ## Parameters

In [ ]:
import requests
import shutil
import os
import yaml
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import uuid

# Notebook parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("osm_url", "https://download.geofabrik.de/north-america/us/massachusetts-latest.osm.pbf")
dbutils.widgets.text("osm_region", "massachusetts")
dbutils.widgets.text("config_path", "")

# Extract parameters
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
osm_url = dbutils.widgets.get("osm_url")
osm_region = dbutils.widgets.get("osm_region")
config_path = dbutils.widgets.get("config_path")

assert catalog and bronze_schema and osm_url, "Missing required parameters: catalog, bronze_schema, osm_url"

# Define paths
osm_filename = osm_url.split('/')[-1]
osm_volume_path = f"/Volumes/{catalog}/{bronze_schema}/osm_data/"
osm_file_path = f"{osm_volume_path}{osm_filename}"
output_table = f"{catalog}.{bronze_schema}.raw_pois"

print(f"Catalog: {catalog}")
print(f"Schema: {bronze_schema}")
print(f"OSM URL: {osm_url}")
print(f"Output table: {output_table}")

In [ ]:
# MAGIC %md
# MAGIC ## Section 1: Download OSM Data

In [ ]:
download_id = str(uuid.uuid4())
download_start = datetime.now()

# Check if file already exists (idempotency)
if os.path.exists(osm_file_path):
    file_size_mb = os.path.getsize(osm_file_path) / (1024 * 1024)
    status = "existing"
    print(f"File already exists: {osm_file_path} ({file_size_mb:.2f} MB)")
else:
    print(f"Downloading {osm_url}...")
    
    # Ensure volume directory exists
    os.makedirs(osm_volume_path, exist_ok=True)
    
    # Download directly to volume
    with requests.get(osm_url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(osm_file_path, "wb") as f:
            shutil.copyfileobj(r.raw, f)
    
    file_size_mb = os.path.getsize(osm_file_path) / (1024 * 1024)
    download_end = datetime.now()
    duration_seconds = (download_end - download_start).total_seconds()
    status = "completed"
    print(f"Download completed: {file_size_mb:.2f} MB in {duration_seconds:.1f} seconds")

# Verify file exists
if not os.path.exists(osm_file_path):
    raise RuntimeError(f"OSM file not found at: {osm_file_path}")

print(f"\nOSM file ready: {osm_file_path}")

In [ ]:
# MAGIC %md
# MAGIC ## Section 2: Extract POIs

In [ ]:
import osmium

# Load configuration if provided
extract_all = True
poi_tag_categories = ['amenity', 'shop', 'leisure', 'tourism', 'office', 'public_transport', 'railway']

if config_path and os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    poi_config = config.get('poi_extraction', {})
    extract_all = poi_config.get('extract_all', True)
    poi_tag_categories = poi_config.get('poi_tag_categories', poi_tag_categories)
    print(f"Loaded config: extract_all={extract_all}, categories={poi_tag_categories}")
else:
    print(f"Using default POI extraction settings")

In [ ]:
class POIHandler(osmium.SimpleHandler):
    """Handler to extract POI nodes from OSM data based on tag categories"""
    
    def __init__(self, extract_all=True, poi_tag_categories=None):
        super().__init__()
        self.pois = []
        
        default_poi_tags = [
            'amenity', 'shop', 'leisure', 'tourism', 'office', 
            'public_transport', 'railway', 'natural', 'building'
        ]
        
        if extract_all:
            self.poi_tag_categories = default_poi_tags
        else:
            self.poi_tag_categories = poi_tag_categories if poi_tag_categories else default_poi_tags
    
    def _has_poi_tag(self, tags):
        """Check if element has any POI tag from the configured categories"""
        tag_keys = {tag.k for tag in tags}
        return any(poi_tag in tag_keys for poi_tag in self.poi_tag_categories)
    
    def node(self, n):
        """Extract nodes with POI tags"""
        if not n.location.valid():
            return
        
        if self._has_poi_tag(n.tags):
            tags_dict = dict(n.tags)
            if tags_dict:
                self.pois.append({
                    'osm_id': str(n.id),
                    'osm_type': 'node',
                    'latitude': n.location.lat,
                    'longitude': n.location.lon,
                    'tags': tags_dict
                })

In [ ]:
# Parse OSM file and extract POIs
print(f"Extracting POIs from {osm_file_path}...")

handler = POIHandler(extract_all=extract_all, poi_tag_categories=poi_tag_categories)
handler.apply_file(osm_file_path)

poi_count = len(handler.pois)
print(f"Extracted {poi_count:,} POIs")

if poi_count == 0:
    raise RuntimeError("No POIs found in OSM file. Check if file contains POI data with matching tags.")

In [ ]:
# MAGIC %md
# MAGIC ## Section 3: Write to Bronze Table

In [ ]:
# Convert POIs to Spark DataFrame
schema = StructType([
    StructField("osm_id", StringType(), False),
    StructField("osm_type", StringType(), False),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("tags", MapType(StringType(), StringType()), True)
])

poi_df = spark.createDataFrame(handler.pois, schema=schema)

# Add ingestion metadata
poi_df = poi_df.withColumn("ingestion_timestamp", F.current_timestamp())

print(f"Created DataFrame with {poi_df.count():,} rows")
display(poi_df.limit(5))

In [ ]:
# Write to Bronze table
(poi_df
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .option("delta.autoOptimize.optimizeWrite", "true")
 .saveAsTable(output_table))

print(f"Written to {output_table}")

In [ ]:
# MAGIC %md
# MAGIC ## Validation

In [ ]:
print("=" * 80)
print("POI INGESTION VALIDATION")
print("=" * 80)

summary = spark.sql(f"""
    SELECT 
        COUNT(*) as total_pois,
        COUNT(DISTINCT osm_id) as unique_pois,
        COUNT(CASE WHEN latitude IS NOT NULL AND longitude IS NOT NULL THEN 1 END) as pois_with_coords,
        COUNT(CASE WHEN tags IS NOT NULL THEN 1 END) as pois_with_tags
    FROM {output_table}
""")
display(summary)

# Sample POI categories
print("\nTop POI categories (from 'amenity' tag):")
spark.sql(f"""
    SELECT tags['amenity'] as amenity, COUNT(*) as count
    FROM {output_table}
    WHERE tags['amenity'] IS NOT NULL
    GROUP BY tags['amenity']
    ORDER BY count DESC
    LIMIT 10
""").show(truncate=False)

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)